# EGC Taxonomy Tree

Reads `.a.phylodist.txt` files from `Metagenome_Samples`, builds the taxonomy tree, and writes one `.tax.json` per sample.

**Run order:** Cell by cell top to bottom. Restart kernel before re-running.

In [1]:
import json
import time
from pathlib import Path
from collections import defaultdict
import sys
sys.path.append(r"C:\Users\sakoehn\Desktop\JGI\Notebooks")
from All_Coverages import allfold
from weighted_totals import calculate_denominator_egc

## 1. Load sample list

In [2]:
with open(r'C:\Users\sakoehn\Desktop\JGI\sample_names.txt') as f:
    samples = [s.strip() for s in f]
print(f"Loaded {len(samples)} samples")

Loaded 128 samples


## 2. Read taxonomy annotation files → tax_db

`tax_db[gene_id]` holds parallel lists: one entry per observation of that gene.

The `lookupId` field is the full semicolon-separated taxonomy path from the last column of the phylodist file.

In [3]:
def read_tax(directpath, filename, s, tax_db):
    """
    Parse one sample's .a.phylodist.txt and append each gene observation
    to tax_db.  Both branches write to the same tax_db parameter.
    """
    path = directpath / filename
    if not path.exists():
        print(f"  WARNING: {path} not found, skipping")
        return

    with open(path, "r") as f:
        for line in f:
            parts   = line.rstrip("\n").split("\t")
            gene_id = parts[0]
            full_path = parts[-1]        # semicolon-separated taxonomy path
            scaff   = gene_id.rsplit('_', 2)[0]

            if gene_id not in tax_db:
                tax_db[gene_id] = {
                    'lookupId':  [full_path],
                    'timepoint': [s],
                    'scaffolds': [scaff],
                }
            else:
                tax_db[gene_id]['lookupId'].append(full_path)
                tax_db[gene_id]['timepoint'].append(s)
                tax_db[gene_id]['scaffolds'].append(scaff)


base   = Path(r"C:\Users\sakoehn\Desktop\JGI\Metagenome_Samples")
tax_db = {}

for s in samples:
    read_tax(base / s, f"{s}.a.phylodist.txt", s, tax_db)

print(f"tax_db: {len(tax_db):,} genes")

tax_db: 71,121,876 genes


In [4]:
# This preview cell is obsolete -- it computed the OLD, per-source-scoped
# gene-weighted denominator (from tax_db alone), before the cross-source
# fix. The real denominator now comes from weighted_totals.py; see the
# "Compute EGC and norm_egc" section below.


## 3. Build taxonomy tree

Flat dict keyed by **lineage prefix**, not by bare taxon name. For each semicolon-separated path, every node in the path is registered and gets the gene + scaffold counted.

Fields per node: `depth` (1-indexed position in path), `category` (root label of the path), `name` (the bare rank label, e.g. "Bacteroidetes", for display), `parent` (the parent's own lineage-prefix key, or `None` for roots).

Key = the lineage truncated to this rank (e.g. `"Bacteria;Bacteroidetes"`), NOT the bare name -- the same name can legitimately appear under more than one ancestry (taxonomic reclassification across samples/pipeline versions; see taxonomy_egc_homonym_bug). Keying by full lineage instead of bare name means two such occurrences get two distinct nodes instead of silently merging into one with a wrong parent.

In [ ]:
tax_tree = {}  # {node_key: {depth, category, name, parent, timepoints}} -- node_key is the full lineage prefix through this rank, e.g. "Bacteria;Bacteroidetes"

start = time.time()

for gene_id, ginfo in tax_db.items():
    lookups    = ginfo.get('lookupId',  [])
    timepoints = ginfo.get('timepoint', [])
    scaffolds  = ginfo.get('scaffolds', [])

    for full_path, sample, scaff in zip(lookups, timepoints, scaffolds):
        raw_parts = full_path.split(';')
        taxpath   = [p.strip() for p in raw_parts if p.strip()]

        # E16 (docs/AUDIT.md): no collapsing of repeated names here, on
        # purpose. This used to remove every consecutive repeated name
        # (not just a trailing one), on the theory that it prevented
        # double-counting -- but node_key below is the FULL lineage
        # PREFIX through each rank, so two consecutive ranks sharing the
        # same name already produce two distinct keys, never the same
        # node twice. The collapse was never load-bearing for that.
        #
        # What it actually did: JGI's phylodist format commonly repeats
        # the deepest RESOLVED rank's name across every rank it couldn't
        # further resolve -- not just once at the very end. Checked
        # directly against all 128 real .a.phylodist.txt files: 7.6M of
        # 71.1M lineage observations (10.7%) have a repeat that ISN'T
        # trailing (e.g. genus AND species both "unclassified
        # Flavobacteriaceae"). Collapsing those silently deleted a real
        # rank position, shifting every rank below it one level too
        # shallow and breaking its match against taxon_paths' fixed
        # 8-position rank columns. Keeping every raw segment, even when
        # a rank's name is identical to its own parent's, keeps rank
        # depth always matching the raw lineage's real position count --
        # an "unclassified X" node whose child is also named
        # "unclassified X" just means JGI couldn't resolve either rank,
        # which is honestly what happened.

        if not taxpath:
            continue

        root_id = taxpath[0]

        for i, tax_id in enumerate(taxpath):
            # node_key is the lineage truncated to THIS rank -- distinct per
            # rank per gene, and identical for two genes that share the same
            # ancestry up to this point. Bare tax_id (the name alone) is only
            # used for display ('name'), never as the dict key -- that's the
            # fix for names that legitimately appear under more than one
            # ancestry (see taxonomy_egc_homonym_bug).
            node_key = ';'.join(taxpath[:i + 1])

            if node_key not in tax_tree:
                tax_tree[node_key] = {
                    'depth':      i + 1,
                    'category':   root_id,
                    'name':       tax_id,
                    'parent':     None if i == 0 else ';'.join(taxpath[:i]),
                    'timepoints': {},
                }

            node = tax_tree[node_key]

            if sample not in node['timepoints']:
                node['timepoints'][sample] = {
                    'egc':       0,
                    'norm_egc':  0.0,
                    'scaffolds': {},
                    'genes':     set(),
                }

            tp = node['timepoints'][sample]
            # Guard matches COG/KEGG/Enzyme: only count this gene's scaffold
            # once per node, even if the same gene_id is observed more than
            # once for this sample (e.g. a duplicate row in .a.phylodist.txt).
            # Without it, a repeated gene_id would inflate 'scaffolds' -- and
            # since every gene is registered at every rank of its lineage,
            # that inflation would propagate from strain all the way up to
            # domain. Verified inert on current data (checked all 128
            # .a.phylodist.txt files, zero gene_ids appear more than once),
            # but added for consistency and to guard against future data.
            if gene_id not in tp['genes']:
                tp['scaffolds'][scaff] = tp['scaffolds'].get(scaff, 0) + 1
            tp['genes'].add(gene_id)

print(f"Taxonomy tree: {len(tax_tree):,} nodes  ({time.time() - start:.1f}s)")

## 4. Compute EGC and norm_egc

`egc = Σ (gene_count_on_scaffold × scaffold_coverage)`, using only genes annotated with a taxonomy hit (from this notebook's own `tax_db`).

`norm_egc = egc / weighted_totals[sample]`, where the denominator comes from `Notebooks/weighted_totals.py` (`calculate_denominator_egc`) and is the SAME value used by `EGC_KEGG_TREE.ipynb` / `EGC_COG_TREE.ipynb` / `EGC_ENZYME_TREE.ipynb` for a given sample -- computed once from every gene with any annotation across all four sources (via the mapping TSVs), not just genes with a taxonomy hit. That shared scope is what makes norm_egc comparable across annotation types on the same plot. norm_egc still stays in (0, 1], since `egc` (a taxonomy-only subset of the shared total) can never exceed it.

In [6]:
# Cross-source-comparable denominator.
#
# calculate_denominator_egc() (Notebooks/weighted_totals.py) computes ONE
# shared total per sample from the mapping TSVs -- every gene annotated by
# ANY of the four sources (taxonomy/KEGG/enzyme/COG), counted once each --
# so this notebook uses the exact same denominator as EGC_COG_TREE.ipynb /
# EGC_ENZYME_TREE.ipynb / EGC_TAX_TREE.ipynb for a given sample. That's what
# makes norm_egc comparable across annotation types on the same plot,
# instead of each notebook normalizing against only its own source's
# annotated genes.
weighted_totals = calculate_denominator_egc(samples)

# Sanity check on first sample: compare against the OLD unweighted,
# not-gene-count-weighted sum (allfold[sample]['total']) as a rough
# magnitude check -- not a correctness check on its own.
_s0 = next(iter(weighted_totals))
_new = weighted_totals[_s0]
_old = allfold.get(_s0, {}).get('total', 0)
print(f'Sample {_s0}: old denom={_old:,.0f}  new denom={_new:,.0f}')
print(f'norm_egc values will be ~{_old/_new:.2f}x smaller, now bounded by 1.0')

Sample 3300060358: old denom=8,298,642  new denom=16,137,978
norm_egc values will be ~0.51x smaller, now bounded by 1.0


In [7]:
def avg_gene_map(node_id):
    node = tax_tree[node_id]
    for sample_name, data in node['timepoints'].items():
        scaffold_counts = data.get('scaffolds')
        if not scaffold_counts or sample_name not in allfold:
            continue

        covs = allfold[sample_name]
        sample_total_depth = weighted_totals[sample_name]

        total_egc = 0
        for scaff, cnt in scaffold_counts.items():
            cov = covs.get(scaff, 0)
            total_egc += cnt * cov

        data['egc']      = total_egc
        data['norm_egc'] = (
            total_egc / sample_total_depth
            if sample_total_depth > 0 else 0.0
        )


for node_id in tax_tree:
    avg_gene_map(node_id)
print('EGC computation complete')


EGC computation complete


## 5. Write .tax.json files

In [ ]:
base = Path(r"C:\Users\sakoehn\Desktop\JGI\DataStructures\Taxonomy")
base.mkdir(parents=True, exist_ok=True)

for s in samples:
    tp_out = {}
    for node_key, node in tax_tree.items():
        if s not in node['timepoints']:
            continue
        tp = node['timepoints'][s]
        tp_out[node_key] = {
            'name':      node['name'],
            'depth':     node['depth'],
            'category':  node['category'],
            'parent':    node['parent'],
            'egc':       tp.get('egc',       0),
            'norm_egc':  tp.get('norm_egc',  0.0),
            'scaffolds': tp.get('scaffolds', {}),
        }
    with open(base / f"{s}.tax.json", 'w') as f:
        json.dump(tp_out, f, indent=2)

print(f"Wrote {len(samples)} .tax.json files to {base}")

Wrote 128 .tax.json files to C:\Users\sakoehn\Desktop\JGI\DataStructures\Taxonomy


: 